# M1 — Data Exploration

Load price data from `data/state.db` and plot closing prices for the 10 sector ETFs.

**Prerequisites:** Run `uv run python scripts/ingest_prices.py` before opening this notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

from config import load_config

DB_PATH = Path('..') / 'data' / 'state.db'
engine = create_engine(f'sqlite:///{DB_PATH}')
universe = load_config('universe')
etf_tickers = universe.ticker_list  # 10 ETFs, excludes SPY benchmark
print('Universe:', etf_tickers)

In [ ]:
df = pd.read_sql(
    text('SELECT date, ticker, close, adj_close FROM prices ORDER BY date'),
    engine,
    parse_dates=['date'],
)
etf_df = df[df['ticker'].isin(etf_tickers)]
print(f'{len(etf_df):,} rows | {etf_df["date"].min().date()} → {etf_df["date"].max().date()}')
etf_df.head()

In [ ]:
# Normalise to 100 at the start of the series for comparability
pivot = etf_df.pivot(index='date', columns='ticker', values='adj_close')
normalised = pivot / pivot.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 7))
for ticker in etf_tickers:
    if ticker in normalised.columns:
        normalised[ticker].plot(ax=ax, label=ticker, linewidth=1.2)

ax.set_title('Sector ETF Adjusted Close — Normalised to 100', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Normalised price (base = 100)')
ax.legend(ncol=2, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Basic summary stats
returns = pivot.pct_change().dropna()
summary = pd.DataFrame({
    'Ann. Return': returns.mean() * 252,
    'Ann. Volatility': returns.std() * (252 ** 0.5),
    'Sharpe (rf=0)': (returns.mean() / returns.std()) * (252 ** 0.5),
    'Max Drawdown': (pivot / pivot.cummax() - 1).min(),
}).round(3)
summary.sort_values('Sharpe (rf=0)', ascending=False)